In [ ]:
import os
import re
from pathlib import Path

import pdfplumber
import fitz  # PyMuPDF
from pprint import pprint

import matplotlib.pyplot as plt

In [ ]:
PDF_PATH = Path("yash_res_swen.pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError("Place a resume PDF in the notebook directory.")

PDF_PATH

In [ ]:
def extract_text_pdfplumber(pdf_path: Path) -> str:
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text.append(page.extract_text() or "")
    return "\n".join(text)

raw_text_plumber = extract_text_pdfplumber(PDF_PATH)
print(raw_text_plumber[:1000])

In [ ]:
def extract_text_pymupdf(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    text = []
    for page in doc:
        text.append(page.get_text())
    return "\n".join(text)

raw_text_mupdf = extract_text_pymupdf(PDF_PATH)
print(raw_text_mupdf[:1000])


In [ ]:
print("=== pdfplumber ===")
print(raw_text_plumber[:500])

print("\n=== PyMuPDF ===")
print(raw_text_mupdf[:500])

In [ ]:

def normalize_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)      # collapse whitespace
    text = text.replace("•", "-")         # normalize bullets
    text = text.replace("●", "-")
    text = text.strip()
    return text


In [ ]:
SECTION_HEADERS = {
    "experience": ["experience", "work experience", "professional experience"],
    "education": ["education"],
    "skills": ["skills"],
    "projects": ["projects"],
    "certifications": ["certifications"],
    "summary": ["summary", "profile", "objective"]
}

# Flatten for matching
ALL_HEADERS = [h for group in SECTION_HEADERS.values() for h in group]

def normalize_header(line: str):
    clean = line.lower().strip()
    for canonical, variants in SECTION_HEADERS.items():
        if any(clean.startswith(v) for v in variants):
            return canonical
    return None


def split_into_sections(text: str):
    sections = {}
    current = "general"
    sections[current] = []

    for line in text.split("\n"):
        header = normalize_header(line)

        if header:
            current = header
            if current not in sections:
                sections[current] = []
        else:
            sections[current].append(line)

    return sections


In [ ]:
def extract_bullets(text: str):
    bullets = re.findall(r"[-•●]\s+(.*)", text)
    return [b.strip() for b in bullets]

bullets = extract_bullets(raw_text_plumber)
pprint(bullets[:10])

In [ ]:
clean_resume_text = normalize_text(raw_text_plumber)
print(clean_resume_text[:1000])
# SAVE CLEANED VERSION
output_path = "resume_cleaned.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(clean_resume_text)

print("Saved cleaned resume to:", output_path)